<a href="https://colab.research.google.com/github/Faizaa01/Deep_Learning_Practices/blob/main/CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader, Subset
from PIL import Image
import kagglehub

In [2]:
path = kagglehub.dataset_download("mohitsingh1804/plantvillage")
print("Path: ", path)

Using Colab cache for faster access to the 'plantvillage' dataset.
Path:  /kaggle/input/plantvillage


In [3]:
#setup

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using Device {device}")

Using Device cuda


In [4]:
Train_Path = os.path.join(path, "PlantVillage", "train")
Val_Path = os.path.join(path, "PlantVillage", "val")

In [6]:
transform = transforms.Compose(
    [
        transforms.Resize((128, 128)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5]*3, std=[0.5]*3)
    ]
)

In [17]:
#custom dataset

class MultiClassClassification(Dataset):
  def __init__(self, root_dir, transform=None):
    super().__init__()

    self.samples = []
    self.transform = transform

    self.classes = sorted([d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, d))])
    self.class_to_idx = {cls_name : idx for idx, cls_name in enumerate(self.classes)}

    for class_name in self.classes:
      class_path = os.path.join(root_dir, class_name)

      for img_name in os.listdir(class_path):
        img_path = os.path.join(class_path, img_name)

        if os.path.isfile(img_path):
          label = self.class_to_idx[class_name]
          self.samples.append((img_path, label))


  def __len__(self):
    return len(self.samples)


  def __getitem__(self, idx):
    img_path, label = self.samples[idx]
    image = Image.open(img_path).convert("RGB")

    if self.transform:
      image = self.transform(image)

    return image, label


In [22]:
train_dataset_full = MultiClassClassification(Train_Path, transform)
test_dataset_full = MultiClassClassification(Val_Path, transform)
num_classes = len(train_dataset_full.classes)

print("Number of classes:", num_classes)
print("Full train size:", len(train_dataset_full))
print("Full test size:", len(test_dataset_full))

Number of classes: 38
Full train size: 43444
Full test size: 10861


In [41]:
train_dataset = Subset(train_dataset_full, list(range(min(200, len(train_dataset_full)))))
test_dataset = Subset(test_dataset_full, list(range(min(200, len(test_dataset_full)))))

In [42]:
pin = True if device.type == 'cuda' else False

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, pin_memory=pin)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, pin_memory=pin)

In [43]:
class MyCNN(nn.Module):
  def __init__(self, num_classes):
    super().__init__()
    self.features = nn.Sequential(
        nn.Conv2d(3, 32, kernel_size=3, padding='same'),
        nn.ReLU(),
        nn.BatchNorm2d(32),
        nn.MaxPool2d(2),

        nn.Conv2d(32, 64, kernel_size=3, padding='same'),
        nn.ReLU(),
        nn.BatchNorm2d(64),
        nn.MaxPool2d(2),

        nn.Conv2d(64, 128, kernel_size=3, padding='same'),
        nn.ReLU(),
        nn.BatchNorm2d(128),
        nn.MaxPool2d(2),
    )

    self.classifier = nn.Sequential(
        nn.Flatten(),
        nn.Linear(128*16*16, 128),
        nn.ReLU(),
        nn.Dropout(0.4),
        nn.Linear(128, 64),
        nn.ReLU(),
        nn.Dropout(0.4),
        nn.Linear(64, num_classes)
    )

  def forward(self, x):
    x = self.features(x)
    x = self.classifier(x)
    return x

In [44]:
model = MyCNN(num_classes=num_classes).to(device)

In [45]:
learning_rate = 0.001
epochs =10
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

In [46]:
for epoch in range(epochs):
  model.train()
  total_loss = 0
  for batch_features, batch_labels in train_loader:
    batch_features = batch_features.to(device)
    batch_labels = batch_labels.to(device)
    outputs = model(batch_features)
    loss = criterion(outputs, batch_labels)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    total_loss += loss.item()

  avg_loss = total_loss /len(train_loader)
  print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")


Epoch 1/10, Loss: 0.5949
Epoch 2/10, Loss: 0.0006
Epoch 3/10, Loss: 0.0155
Epoch 4/10, Loss: 0.0000
Epoch 5/10, Loss: 0.0000
Epoch 6/10, Loss: 0.0020
Epoch 7/10, Loss: 0.0000
Epoch 8/10, Loss: 0.0000
Epoch 9/10, Loss: 0.0000
Epoch 10/10, Loss: 0.0000


In [47]:
def evaluate(loader):
  model.eval()
  total, correct = 0, 0
  with torch.no_grad():
    for batch_features, batch_labels in loader:
      batch_features = batch_features.to(device)
      batch_labels = batch_labels.to(device)
      outputs = model(batch_features)
      _, predicted = torch.max(outputs, 1)
      total += batch_labels.size(0)
      correct += (predicted == batch_labels).sum().item()
  return correct/total

In [48]:
test_acc = evaluate(test_loader)
print("Test Accuracy: ", test_acc)

Test Accuracy:  0.63
